# HEAR: KcELECTRA Binary + Hate Type Pipeline

이 노트북은 Colab GPU에서 바로 실행할 수 있도록 만든 전체 파이프라인입니다.

1. Google Drive 프로젝트 폴더로 이동
2. 패키지 설치
3. binary 유해표현 모델 학습
4. CSV 전체 binary 예측
5. 유해 데이터만 기반으로 혐오 유형 멀티라벨 모델 학습
6. binary 예측 결과에 혐오 유형 예측 컬럼 추가
7. 결과 zip 다운로드

먼저 Colab 메뉴에서 `런타임 > 런타임 유형 변경 > GPU`를 선택하세요.

In [ ]:
# 1. Drive mount and project path
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# TODO: Drive에 올린 프로젝트 경로에 맞게 필요하면 수정하세요.
PROJECT_DIR = Path('/content/drive/MyDrive/HEAR/code')

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'프로젝트 폴더를 찾을 수 없습니다: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print('cwd =', Path.cwd())

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements-hate-detector.txt

In [ ]:
# 3. Environment check
import torch, transformers, datasets, sklearn

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 4. Pipeline settings
# 이미 학습된 모델이 있으면 TRAIN_BINARY 또는 TRAIN_HATE_TYPE을 False로 바꿔 재사용할 수 있습니다.

TRAIN_BINARY = True
TUNE_BINARY_THRESHOLD = True
PREDICT_BINARY_DIR = True
TRAIN_HATE_TYPE = True
PREDICT_HATE_TYPE_DIR = True

# 코드 실행만 빠르게 확인하고 싶을 때 True로 바꾸세요.
RUN_SMOKE_TEST = False

BINARY_MODEL_DIR = 'models/kc-electra-komultitext-binary'
HATE_TYPE_MODEL_DIR = 'models/kc-electra-komultitext-hate-type'

INPUT_CSV_DIR = 'crawler/dc'
BINARY_OUTPUT_DIR = '분석 데이터/hate_predictions'
HATE_TYPE_OUTPUT_DIR = '분석 데이터/hate_type_predictions'

BINARY_EPOCHS = 3
HATE_TYPE_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
PREDICT_BATCH_SIZE = 64
MAX_LENGTH_TRAIN = 160
MAX_LENGTH_PREDICT = 192

print('binary model:', BINARY_MODEL_DIR)
print('hate type model:', HATE_TYPE_MODEL_DIR)
print('input csv dir:', INPUT_CSV_DIR)

In [ ]:
# 5. Helper runner
import subprocess
import shlex

def run_cmd(cmd):
    print('\n$ ' + ' '.join(shlex.quote(str(part)) for part in cmd))
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')

def path_exists(path):
    return Path(path).exists()

In [ ]:
# 6. Optional smoke tests
if RUN_SMOKE_TEST:
    run_cmd([
        'python', '-m', 'hate_detector.train_binary',
        '--output-dir', 'models/smoke-kc-electra-binary',
        '--epochs', '0.01',
        '--max-train-samples', '16',
        '--max-eval-samples', '16',
        '--batch-size', '4',
        '--max-length', '64',
    ])
    run_cmd([
        'python', '-m', 'hate_detector.train_hate_type',
        '--output-dir', 'models/smoke-kc-electra-hate-type',
        '--epochs', '0.01',
        '--max-train-samples', '16',
        '--max-eval-samples', '16',
        '--batch-size', '4',
        '--max-length', '64',
    ])
else:
    print('Smoke tests skipped.')

In [ ]:
# 7. Train binary harmful-expression model
if TRAIN_BINARY:
    run_cmd([
        'python', '-m', 'hate_detector.train_binary',
        '--output-dir', BINARY_MODEL_DIR,
        '--epochs', str(BINARY_EPOCHS),
        '--batch-size', str(TRAIN_BATCH_SIZE),
        '--max-length', str(MAX_LENGTH_TRAIN),
        '--harmful-class-weight', '1.8',
    ])
else:
    print('Binary training skipped.')

In [ ]:
# 8. Tune binary threshold for high recall
if TUNE_BINARY_THRESHOLD:
    run_cmd([
        'python', '-m', 'hate_detector.tune_threshold',
        '--model-dir', BINARY_MODEL_DIR,
        '--min-recall', '0.90',
        '--batch-size', str(PREDICT_BATCH_SIZE),
        '--max-length', str(MAX_LENGTH_TRAIN),
    ])
else:
    print('Threshold tuning skipped.')

import json
threshold_path = Path(BINARY_MODEL_DIR) / 'threshold.json'
if threshold_path.exists():
    with threshold_path.open(encoding='utf-8') as f:
        print(json.dumps(json.load(f), ensure_ascii=False, indent=2))

In [ ]:
# 9. Predict harmful rows for all CSV files
if PREDICT_BINARY_DIR:
    run_cmd([
        'python', '-m', 'hate_detector.predict_dir',
        '--model-dir', BINARY_MODEL_DIR,
        '--input-dir', INPUT_CSV_DIR,
        '--output-dir', BINARY_OUTPUT_DIR,
        '--exclude-name', 'all_schools.csv',
        '--batch-size', str(PREDICT_BATCH_SIZE),
        '--max-length', str(MAX_LENGTH_PREDICT),
    ])
else:
    print('Binary directory prediction skipped.')

In [ ]:
# 10. Check binary prediction summary
import pandas as pd

binary_summary_path = Path(BINARY_OUTPUT_DIR) / 'summary.csv'
if binary_summary_path.exists():
    binary_summary = pd.read_csv(binary_summary_path)
    display(binary_summary)
else:
    print('No binary summary found:', binary_summary_path)

In [ ]:
# 11. Train hate-type multi-label model
if TRAIN_HATE_TYPE:
    run_cmd([
        'python', '-m', 'hate_detector.train_hate_type',
        '--output-dir', HATE_TYPE_MODEL_DIR,
        '--epochs', str(HATE_TYPE_EPOCHS),
        '--batch-size', str(TRAIN_BATCH_SIZE),
        '--max-length', str(MAX_LENGTH_TRAIN),
        '--threshold', '0.5',
    ])
else:
    print('Hate-type training skipped.')

In [ ]:
# 12. Check hate-type label config
import json

label_config_path = Path(HATE_TYPE_MODEL_DIR) / 'label_config.json'
if label_config_path.exists():
    with label_config_path.open(encoding='utf-8') as f:
        print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
else:
    print('No label config found:', label_config_path)

In [ ]:
# 13. Add hate-type predictions to harmful rows
# 기본값은 is_harmful=1인 행에만 유형 모델을 적용합니다.
if PREDICT_HATE_TYPE_DIR:
    run_cmd([
        'python', '-m', 'hate_detector.predict_hate_type_dir',
        '--type-model-dir', HATE_TYPE_MODEL_DIR,
        '--input-dir', BINARY_OUTPUT_DIR,
        '--output-dir', HATE_TYPE_OUTPUT_DIR,
        '--batch-size', str(PREDICT_BATCH_SIZE),
        '--max-length', str(MAX_LENGTH_PREDICT),
    ])
else:
    print('Hate-type directory prediction skipped.')

In [ ]:
# 14. Check hate-type summary and examples
from pathlib import Path
import pandas as pd

type_summary_path = Path(HATE_TYPE_OUTPUT_DIR) / 'summary.csv'
if type_summary_path.exists():
    type_summary = pd.read_csv(type_summary_path)
    display(type_summary)
else:
    print('No type summary found:', type_summary_path)

files = sorted(Path(HATE_TYPE_OUTPUT_DIR).glob('*_hate.csv'))
if files:
    all_typed = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
    preview_cols = [
        'source_file', '제목', '내용', 'harmful_score',
        'primary_hate_type', 'matched_hate_types'
    ]
    preview_cols = [col for col in preview_cols if col in all_typed.columns]
    display(all_typed.sort_values('harmful_score', ascending=False)[preview_cols].head(30))

    type_cols = [col for col in all_typed.columns if col.startswith('hate_type_') and not col.endswith('_score')]
    if type_cols:
        display(all_typed[type_cols].sum().sort_values(ascending=False).to_frame('count'))
else:
    print('No typed CSV files found:', HATE_TYPE_OUTPUT_DIR)

In [ ]:
# 15. Zip results
RESULT_ZIP = 'hate_type_results.zip'

!zip -r "$RESULT_ZIP" \
  "$BINARY_OUTPUT_DIR" \
  "$HATE_TYPE_OUTPUT_DIR" \
  "$BINARY_MODEL_DIR/threshold.json" \
  "$BINARY_MODEL_DIR/training_metrics.json" \
  "$HATE_TYPE_MODEL_DIR/label_config.json" \
  "$HATE_TYPE_MODEL_DIR/training_metrics.json"

print('created:', RESULT_ZIP)

In [ ]:
# 16. Download zip
from google.colab import files
files.download(RESULT_ZIP)